# Global Seismic Trends: Data-Driven Earthquake Insights
## 01 — Data Collection

**Data source:** https://earthquake.usgs.gov/fdsnws/event/1/

1. Configure project paths and test API connectivity.
2. Inspect an example GeoJSON event and its metadata.
3. Extract the 26 core features and retain selected extra API properties.
4. Retrieve a rolling five-year period in monthly windows.
5. Automatically split oversized requests into smaller time windows.
6. Validate IDs, timestamps, coordinates, and missing values.



## 1. Imports and project paths



In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time

import requests
import pandas as pd
import numpy as np
from IPython.display import display

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
REQUEST_TIMEOUT_SECONDS = 90
MAX_RETRIES = 4
REQUEST_PAUSE_SECONDS = 0.15
API_RESULT_LIMIT = 20000

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data directory:", RAW_DATA_DIR.resolve())
print("Imports and paths configured successfully.")


Project root: D:\global seismic trends
Raw data directory: D:\global seismic trends\data\raw
Imports and paths configured successfully.


## 2. Test the USGS API

In [4]:
test_params = {
    "format": "geojson",
    "starttime": "2025-01-01T00:00:00",
    "endtime": "2025-02-01T00:00:00",
    "orderby": "time-asc",
    "limit": API_RESULT_LIMIT,
}

response = requests.get(API_URL, params=test_params, timeout=REQUEST_TIMEOUT_SECONDS)
print("HTTP status:", response.status_code)
response.raise_for_status()

test_payload = response.json()
test_features = test_payload.get("features", [])
test_metadata = test_payload.get("metadata", {})

print("API title:", test_metadata.get("title"))
print("API reported count:", test_metadata.get("count"))
print("Records received:", len(test_features))
print("Request URL:", response.url)


HTTP status: 200
API title: USGS Earthquakes
API reported count: None
Records received: 11631
Request URL: https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2025-01-01T00%3A00%3A00&endtime=2025-02-01T00%3A00%3A00&orderby=time-asc&limit=20000


In [5]:
if test_features:
    first_event = test_features[0]
    print("First event ID:", first_event.get("id"))
    print("\nFirst event properties:")
    display(pd.Series(first_event.get("properties") or {}, name="value"))
    print("\nCoordinates [longitude, latitude, depth_km]:")
    print((first_event.get("geometry") or {}).get("coordinates"))
else:
    print("No events were returned for the test period.")


First event ID: nn00891066

First event properties:


mag                                                      1.2
place                         38 km SSE of Goldfield, Nevada
time                                           1735689730026
updated                                        1738957575907
tz                                                      None
url        https://earthquake.usgs.gov/earthquakes/eventp...
detail     https://earthquake.usgs.gov/fdsnws/event/1/que...
felt                                                    None
cdi                                                     None
mmi                                                     None
alert                                                   None
status                                              reviewed
tsunami                                                    0
sig                                                       22
net                                                       nn
code                                                00891066
ids                     


Coordinates [longitude, latitude, depth_km]:
[-117.1341, 37.3688, 6.7]


## 3. Define the dataset schema and extraction function


In [6]:
CORE_COLUMNS = [
    "id", "time", "updated", "latitude", "longitude", "depth_km",
    "mag", "magType", "place", "status", "tsunami", "sig", "net",
    "nst", "dmin", "rms", "gap", "magError", "depthError", "magNst",
    "locationSource", "magSource", "types", "ids", "sources", "type"
]

EXTRA_COLUMNS = [
    "alert", "felt", "cdi", "mmi", "url", "detail", "place_type", "time_zone"
]

ALL_COLUMNS = CORE_COLUMNS + EXTRA_COLUMNS

def extract_event(feature: dict) -> dict:
    """Flatten one USGS GeoJSON feature into the project schema."""
    properties = feature.get("properties") or {}
    geometry = feature.get("geometry") or {}
    coordinates = geometry.get("coordinates") or []

    event = {
        "id": feature.get("id"),
        "time": properties.get("time"),
        "updated": properties.get("updated"),
        "longitude": coordinates[0] if len(coordinates) > 0 else None,
        "latitude": coordinates[1] if len(coordinates) > 1 else None,
        "depth_km": coordinates[2] if len(coordinates) > 2 else None,
    }

    for column in CORE_COLUMNS:
        if column not in event:
            event[column] = properties.get(column)

    for column in EXTRA_COLUMNS:
        event[column] = properties.get(column)

    # USGS timestamps are Unix epoch milliseconds.
    for column in ("time", "updated"):
        value = event.get(column)
        event[column] = (
            pd.to_datetime(value, unit="ms", utc=True, errors="coerce")
            if value is not None else pd.NaT
        )

    # Preserve list/dictionary values in a CSV-compatible form.
    for column in ("types", "ids", "sources"):
        value = event.get(column)
        if isinstance(value, (list, dict)):
            event[column] = json.dumps(value, ensure_ascii=False)

    return event

test_df = pd.DataFrame([extract_event(feature) for feature in test_features])
for column in ALL_COLUMNS:
    if column not in test_df.columns:
        test_df[column] = pd.NA
test_df = test_df[ALL_COLUMNS]

print("Test DataFrame shape:", test_df.shape)
print("Core feature count:", len(CORE_COLUMNS))
display(test_df.head())


Test DataFrame shape: (11631, 34)
Core feature count: 26


,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,sources,type,alert,felt,cdi,mmi,url,detail,place_type,time_zone
0,nn00891066,2025-01-01 00:02:10.026+00:00,2025-02-07 19:46:15.907+00:00,37.3688,-117.1341,6.7000,1.2,ml,"38 km SSE of Goldfield, Nevada",reviewed,...,",nn,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
1,nn00891068,2025-01-01 00:04:51.942+00:00,2025-02-07 22:20:26.114+00:00,37.3560,-117.1370,7.0000,1.3,ml,"40 km SSE of Goldfield, Nevada",reviewed,...,",nn,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
2,ak0251nkqz1,2025-01-01 00:06:53.827+00:00,2025-01-28 16:40:27.643+00:00,64.9821,-147.4018,0.0000,1.2,ml,"10 km ENE of Fox, Alaska",reviewed,...,",ak,",explosion,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
3,tx2025aaak,2025-01-01 00:10:38.599+00:00,2025-01-02 17:31:35.860+00:00,31.5230,-103.6620,4.9976,1.7,ml,"19 km NW of Pecos, Texas",reviewed,...,",tx,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
4,us6000pjr6,2025-01-01 00:13:22.650+00:00,2025-03-17 21:33:57.040+00:00,12.7441,143.6244,35.0000,4.2,mb,"127 km WSW of Merizo Village, Guam",reviewed,...,",us,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None


## 4. Configure the five-year collection window


In [ ]:
END_DATETIME = pd.Timestamp.now(tz="UTC").floor("s")
START_DATETIME = END_DATETIME - pd.DateOffset(years=5)

if START_DATETIME >= END_DATETIME:
    raise ValueError("START_DATETIME must be earlier than END_DATETIME.")

print("Collection start (UTC):", START_DATETIME)
print("Collection end (UTC):", END_DATETIME)


Collection start (UTC): 2021-10-23 13:34:56+00:00
Collection end (UTC): 2026-10-23 13:34:56+00:00


## 5. API retrieval with automatic interval splitting


In [8]:
def month_windows(start: pd.Timestamp, end: pd.Timestamp):
    """Yield consecutive month-sized windows clipped to [start, end)."""
    cursor = start
    while cursor < end:
        next_month = cursor.normalize().replace(day=1) + pd.offsets.MonthBegin(1)
        window_end = min(next_month, end)
        yield cursor, window_end
        cursor = window_end


def fetch_interval(start: pd.Timestamp, end: pd.Timestamp, session: requests.Session,
                   depth: int = 0, min_interval_seconds: int = 1) -> list:
    """Fetch one interval; split it automatically if the response may be capped."""
    params = {
        "format": "geojson",
        "starttime": start.strftime("%Y-%m-%dT%H:%M:%S"),
        "endtime": end.strftime("%Y-%m-%dT%H:%M:%S"),
        "orderby": "time-asc",
        "limit": API_RESULT_LIMIT,
    }

    payload = None
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(API_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            payload = response.json()
            break
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                wait_seconds = min(2 ** (attempt - 1), 8)
                print(
                    f"Retry {attempt}/{MAX_RETRIES - 1} for "
                    f"{start.isoformat()} to {end.isoformat()}: {exc}"
                )
                time.sleep(wait_seconds)

    if payload is None:
        raise RuntimeError(
            f"API request failed for interval {start} to {end}"
        ) from last_error

    features = payload.get("features", [])
    reported_count = (payload.get("metadata") or {}).get("count")

    possible_truncation = (
        len(features) >= API_RESULT_LIMIT
        or (isinstance(reported_count, int) and reported_count > len(features))
    )

    if possible_truncation:
        duration_seconds = (end - start).total_seconds()
        if duration_seconds <= min_interval_seconds:
            raise RuntimeError(
                f"Potentially truncated response even at the smallest interval: "
                f"{start} to {end}. Reduce API_RESULT_LIMIT or inspect the API response."
            )

        midpoint = start + (end - start) / 2
        midpoint = midpoint.floor("s")

        if midpoint <= start or midpoint >= end:
            raise RuntimeError(
                f"Unable to split potentially truncated interval {start} to {end}."
            )

        print(
            f"Splitting interval {start:%Y-%m-%d %H:%M:%S} to "
            f"{end:%Y-%m-%d %H:%M:%S}: received {len(features):,} records "
            f"(reported count: {reported_count})"
        )

        left_features = fetch_interval(start, midpoint, session, depth + 1)
        right_features = fetch_interval(midpoint, end, session, depth + 1)
        return left_features + right_features

    time.sleep(REQUEST_PAUSE_SECONDS)
    return features


print("API retrieval and automatic splitting functions are ready.")


API retrieval and automatic splitting functions are ready.


## 6. Retrieve the five-year dataset


In [9]:
all_features = []
collection_log = []

with requests.Session() as session:
    session.headers.update({
        "User-Agent": "GlobalSeismicTrends-StudentProject/1.0 (educational data analysis)"
    })

    windows = list(month_windows(START_DATETIME, END_DATETIME))
    print(f"Monthly windows to retrieve: {len(windows)}")

    for index, (window_start, window_end) in enumerate(windows, start=1):
        features = fetch_interval(window_start, window_end, session)
        all_features.extend(features)

        collection_log.append({
            "window_start_utc": window_start.isoformat(),
            "window_end_utc": window_end.isoformat(),
            "records_received_including_boundary_duplicates": len(features),
        })

        print(
            f"[{index}/{len(windows)}] {window_start:%Y-%m-%d} to "
            f"{window_end:%Y-%m-%d}: {len(features):,} records"
        )

collection_log_df = pd.DataFrame(collection_log)
print("\nTotal features before de-duplication:", f"{len(all_features):,}")
display(collection_log_df.head())


Monthly windows to retrieve: 61
[1/61] 2021-10-23 to 2021-11-01: 3,375 records
[2/61] 2021-11-01 to 2021-12-01: 11,583 records
[3/61] 2021-12-01 to 2022-01-01: 11,400 records
[4/61] 2022-01-01 to 2022-02-01: 12,954 records
[5/61] 2022-02-01 to 2022-03-01: 11,768 records
[6/61] 2022-03-01 to 2022-04-01: 12,803 records
[7/61] 2022-04-01 to 2022-05-01: 12,430 records
[8/61] 2022-05-01 to 2022-06-01: 12,167 records
[9/61] 2022-06-01 to 2022-07-01: 11,851 records
[10/61] 2022-07-01 to 2022-08-01: 12,231 records
[11/61] 2022-08-01 to 2022-09-01: 13,019 records
[12/61] 2022-09-01 to 2022-10-01: 12,996 records
[13/61] 2022-10-01 to 2022-11-01: 13,066 records
[14/61] 2022-11-01 to 2022-12-01: 12,647 records
[15/61] 2022-12-01 to 2023-01-01: 12,808 records
[16/61] 2023-01-01 to 2023-02-01: 12,199 records
[17/61] 2023-02-01 to 2023-03-01: 11,754 records
[18/61] 2023-03-01 to 2023-04-01: 15,792 records
[19/61] 2023-04-01 to 2023-05-01: 13,349 records
[20/61] 2023-05-01 to 2023-06-01: 12,823 record

,window_start_utc,window_end_utc,records_received_including_boundary_duplicates
0,2021-10-23T13:34:56+00:00,2021-11-01T00:00:00+00:00,3375
1,2021-11-01T00:00:00+00:00,2021-12-01T00:00:00+00:00,11583
2,2021-12-01T00:00:00+00:00,2022-01-01T00:00:00+00:00,11400
3,2022-01-01T00:00:00+00:00,2022-02-01T00:00:00+00:00,12954
4,2022-02-01T00:00:00+00:00,2022-03-01T00:00:00+00:00,11768


## 7. Flatten records and remove duplicate event ID

In [10]:
records = [extract_event(feature) for feature in all_features]
earthquakes_raw = pd.DataFrame(records)

for column in ALL_COLUMNS:
    if column not in earthquakes_raw.columns:
        earthquakes_raw[column] = pd.NA
earthquakes_raw = earthquakes_raw[ALL_COLUMNS]

rows_before_dedup = len(earthquakes_raw)
earthquakes_raw = earthquakes_raw.drop_duplicates(subset=["id"], keep="last").reset_index(drop=True)
rows_after_dedup = len(earthquakes_raw)

print("Rows before de-duplication:", f"{rows_before_dedup:,}")
print("Rows after de-duplication:", f"{rows_after_dedup:,}")
print("Duplicate rows removed:", f"{rows_before_dedup - rows_after_dedup:,}")
print("Final dataset shape:", earthquakes_raw.shape)
display(earthquakes_raw.head())


Rows before de-duplication: 739,587
Rows after de-duplication: 739,587
Duplicate rows removed: 0
Final dataset shape: (739587, 34)


,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,sources,type,alert,felt,cdi,mmi,url,detail,place_type,time_zone
0,ak021dlr1b5t,2021-10-23 13:37:08.179+00:00,2021-12-06 18:53:10.237+00:00,62.252900,-150.1420,20.00,1.00,ml,"8 km SSW of Talkeetna, Alaska",reviewed,...,",ak,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
1,ak021dlr50qf,2021-10-23 13:54:39.139+00:00,2021-12-06 18:53:35.829+00:00,67.322400,-156.2287,0.20,1.60,ml,northern Alaska,reviewed,...,",ak,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
2,hv72766952,2021-10-23 13:56:09.690+00:00,2021-10-28 22:52:44.660+00:00,19.406333,-155.6660,2.74,2.42,ml,"21 km ESE of Honaunau-Napoopoo, Hawaii",reviewed,...,",hv,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
3,ak021dlrhf73,2021-10-23 14:12:25.437+00:00,2021-12-06 18:53:09.794+00:00,59.525400,-139.4311,10.00,1.60,ml,"16 km E of Yakutat, Alaska",reviewed,...,",ak,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None
4,us7000fpgu,2021-10-23 14:16:02.236+00:00,2022-01-08 17:43:29.040+00:00,51.152000,97.9311,10.00,4.20,mb,"67 km NNE of Kungurtug, Russia",reviewed,...,",us,",earthquake,NaN,NaN,NaN,NaN,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,None,None


## 8. Validate coverage and data quality


In [11]:
validation_summary = {
    "row_count": len(earthquakes_raw),
    "unique_event_ids": earthquakes_raw["id"].nunique(dropna=True),
    "duplicate_ids_remaining": earthquakes_raw["id"].duplicated().sum(),
    "missing_event_ids": earthquakes_raw["id"].isna().sum(),
    "missing_timestamps": earthquakes_raw["time"].isna().sum(),
    "earliest_event_time_utc": str(earthquakes_raw["time"].min()),
    "latest_event_time_utc": str(earthquakes_raw["time"].max()),
    "latitude_outside_valid_range": (
        ((earthquakes_raw["latitude"] < -90) | (earthquakes_raw["latitude"] > 90)).fillna(False)
    ).sum(),
    "longitude_outside_valid_range": (
        ((earthquakes_raw["longitude"] < -180) | (earthquakes_raw["longitude"] > 180)).fillna(False)
    ).sum(),
    "negative_depth_records_flagged": (earthquakes_raw["depth_km"] < 0).fillna(False).sum(),
}

display(pd.Series(validation_summary, name="value").to_frame())

print("\nMissing values in core fields:")
display(
    earthquakes_raw[CORE_COLUMNS]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

print("\nNumeric summary:")
display(
    earthquakes_raw[["mag", "depth_km", "latitude", "longitude", "sig"]]
    .describe()
)


,value
row_count,739587
unique_event_ids,739587
duplicate_ids_remaining,0
missing_event_ids,0
missing_timestamps,0
earliest_event_time_utc,2021-10-23 13:37:08.179000+00:00
latest_event_time_utc,2026-10-23 13:17:43.510000+00:00
latitude_outside_valid_range,0
longitude_outside_valid_range,0
negative_depth_records_flagged,25045



Missing values in core fields:


,missing_count
locationSource,739587
magError,739587
depthError,739587
magNst,739587
magSource,739587
dmin,195926
nst,158697
gap,147926
rms,57
mag,56



Numeric summary:


,mag,depth_km,latitude,longitude,sig
count,739531.000000,739587.000000,739587.000000,739587.000000,739587.000000
mean,1.658084,24.517510,39.914615,-110.106390,68.379264
std,1.298342,55.649821,19.974530,75.789729,98.682717
min,-5.000000,-10.000000,-84.493200,-179.999700,0.000000
25%,0.850000,3.483643,33.534000,-152.255000,11.000000
50%,1.400000,8.357000,38.824333,-122.765663,30.000000
75%,2.100000,18.100000,55.866100,-112.905500,68.000000
max,8.800000,683.578000,87.081500,179.999400,2910.000000


## 9. Check unavailable or fully missing fields


In [12]:
fully_missing_core_fields = [
    column for column in CORE_COLUMNS
    if earthquakes_raw[column].isna().all()
]

print("Core fields with no non-missing values:")
if fully_missing_core_fields:
    for field in fully_missing_core_fields:
        print("-", field)
else:
    print("None")

print("\nInterpretation:")
print(
    "These fields are retained to preserve the requested schema. "
    "Their absence in this response does not mean their values are zero."
)


Core fields with no non-missing values:
- magError
- depthError
- magNst
- locationSource
- magSource

Interpretation:
These fields are retained to preserve the requested schema. Their absence in this response does not mean their values are zero.


## 10. Save raw data, collection log, and metadata

In [13]:
run_timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")

csv_path = RAW_DATA_DIR / f"usgs_earthquakes_{run_timestamp}.csv"
json_path = RAW_DATA_DIR / f"usgs_earthquakes_{run_timestamp}.json"
log_path = RAW_DATA_DIR / f"usgs_collection_log_{run_timestamp}.csv"
metadata_path = RAW_DATA_DIR / f"usgs_collection_metadata_{run_timestamp}.json"

earthquakes_raw.to_csv(csv_path, index=False)
earthquakes_raw.to_json(json_path, orient="records", date_format="iso", indent=2)
collection_log_df.to_csv(log_path, index=False)

run_metadata = {
    "source": API_URL,
    "retrieved_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "start_datetime_utc": START_DATETIME.isoformat(),
    "end_datetime_utc": END_DATETIME.isoformat(),
    "row_count": int(len(earthquakes_raw)),
    "unique_event_ids": int(earthquakes_raw["id"].nunique(dropna=True)),
    "core_feature_count": len(CORE_COLUMNS),
    "core_columns": CORE_COLUMNS,
    "extra_columns": EXTRA_COLUMNS,
    "deduplication_key": "id",
    "fully_missing_core_fields": fully_missing_core_fields,
    "validation_summary": {
        key: int(value) if isinstance(value, (np.integer,)) else value
        for key, value in validation_summary.items()
    },
    "notes": [
        "USGS catalog data can be revised after retrieval.",
        "Missing values have not been imputed.",
        "Automatic splitting is triggered by a response at the configured result limit or a reported count greater than returned records.",
        "If the API does not report a usable count and truncates below the configured limit without signaling it, completeness cannot be guaranteed solely from this response.",
        "Country, continent, casualty, and economic-loss fields require additional sources or geographic processing."
    ],
}

metadata_path.write_text(json.dumps(run_metadata, indent=2, default=str), encoding="utf-8")

print("Saved output files:")
for path in [csv_path, json_path, log_path, metadata_path]:
    print(path)


Saved output files:
d:\global seismic trends\data\raw\usgs_earthquakes_20261023T134313Z.csv
d:\global seismic trends\data\raw\usgs_earthquakes_20261023T134313Z.json
d:\global seismic trends\data\raw\usgs_collection_log_20261023T134313Z.csv
d:\global seismic trends\data\raw\usgs_collection_metadata_20261023T134313Z.json


## 11. Final summary

In [14]:
final_summary = {
    "collection_start_utc": str(START_DATETIME),
    "collection_end_utc": str(END_DATETIME),
    "unique_earthquakes": int(earthquakes_raw["id"].nunique(dropna=True)),
    "dataset_columns": len(earthquakes_raw.columns),
    "core_features": len(CORE_COLUMNS),
    "extra_fields": len(EXTRA_COLUMNS),
    "duplicate_ids_remaining": int(earthquakes_raw["id"].duplicated().sum()),
    "missing_ids": int(earthquakes_raw["id"].isna().sum()),
    "missing_timestamps": int(earthquakes_raw["time"].isna().sum()),
    "fully_missing_core_fields": ", ".join(fully_missing_core_fields) or "None",
    "csv_output": str(csv_path),
}
display(pd.Series(final_summary, name="value").to_frame())

print("\nNotebook 01 data collection finished.")
print("Next notebook: 02_data_cleaning.ipynb")


,value
collection_start_utc,2021-10-23 13:34:56+00:00
collection_end_utc,2026-10-23 13:34:56+00:00
unique_earthquakes,739587
dataset_columns,34
core_features,26
extra_fields,8
duplicate_ids_remaining,0
missing_ids,0
missing_timestamps,0
fully_missing_core_fields,"magError, depthError, magNst, locationSource, ..."



Notebook 01 data collection finished.
Next notebook: 02_data_cleaning.ipynb
